# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/areebahassann/Starter_Notebook/blob/main/work/notebooks/w03_data_contract.ipynb)

**Lane: Page/URL lane**  
**Author: Areeba Hassan**

In [ ]:
# Setup — install DuckDB and load the warehouse
!pip install duckdb --quiet

import duckdb
import pandas as pd

# Connect to the FlyRank internship warehouse (HuggingFace hosted)
con = duckdb.connect()

# Load the anonymized dataset from the repo
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/areebahassann/Starter_Notebook/main/data/raw/content_refresh_anonymized.csv',
    'content_refresh_anonymized.csv'
)

# Register as a DuckDB table
con.execute("CREATE TABLE pages AS SELECT * FROM read_csv_auto('content_refresh_anonymized.csv')")
print('Data loaded. Shape:', con.execute('SELECT COUNT(*), COUNT(DISTINCT month) FROM pages').fetchone())

## 1. Unit of analysis + time window

**One row = one page (URL) observed in one calendar month.**

- The grain is `(page_id, month)` — one unique page measured once per month.
- Time window used in this contract: `month = 2026-03` (a mid-panel month with full data coverage).
- The full panel spans multiple months; we verify the exact range below.

In [ ]:
# Verify grain and time window
con.execute("""
    SELECT
        MIN(month)   AS earliest_month,
        MAX(month)   AS latest_month,
        COUNT(*)     AS total_rows,
        COUNT(DISTINCT month) AS distinct_months
    FROM pages
""").df()

## 2. Fields: feature / label / context / excluded

### Label (what we predict)
| Field | Role | Notes |
|---|---|---|
| `needs_refresh` | **Label** | Boolean — TRUE if page needs a content update. This is what we predict. |

### Features (inputs to the model — knowable BEFORE the decision)
| Field | Role | Notes |
|---|---|---|
| `avg_position` | Feature | Mean search ranking position over the month |
| `clicks` | Feature | Total GSC clicks in the month |
| `impressions` | Feature | Total GSC impressions in the month |
| `ctr` | Feature | Click-through rate (clicks / impressions) |
| `word_count` | Feature | Length of the page content |

### Context (used for filtering/grouping, not for model input)
| Field | Role | Notes |
|---|---|---|
| `month` | Context | Time window identifier |
| `page_id` | Context | Anonymous page identifier |
| `available` | Context | Whether the page had GSC data — used as a filter |

### Excluded
| Field | Why excluded |
|---|---|
| Any raw URL / title / keyword | Contains client-identifying information — excluded per DATA_USE.md |
| Future month metrics | Not knowable at decision time — would cause leakage |

In [ ]:
# Show all columns available in the dataset
con.execute("DESCRIBE pages").df()

## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim in Section 1 & 2 gets a query here. A contract claim without a query is a guess.

In [ ]:
# QUERY 1: Verify grain — (page_id, month) should be unique (no duplicates)
result = con.execute("""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT page_id || '|' || CAST(month AS VARCHAR)) AS unique_page_month_pairs
    FROM pages
    WHERE month = '2026-03'
""").df()

print("Grain check for month=2026-03:")
print(result)
print("\n✅ Grain is clean if total_rows == unique_page_month_pairs")

In [ ]:
# QUERY 2: Row count and date span for our slice (month=2026-03)
con.execute("""
    SELECT
        month,
        COUNT(*)                             AS row_count,
        COUNT(DISTINCT page_id)              AS distinct_pages,
        SUM(CASE WHEN needs_refresh THEN 1 ELSE 0 END) AS pages_needing_refresh
    FROM pages
    WHERE month = '2026-03'
    GROUP BY month
""").df()

In [ ]:
# QUERY 3: Availability filter — how many rows survive IS TRUE check
con.execute("""
    SELECT
        COUNT(*)                                    AS total_rows_in_month,
        SUM(CASE WHEN available IS TRUE THEN 1 ELSE 0 END)  AS available_rows,
        ROUND(
            100.0 * SUM(CASE WHEN available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1
        )                                           AS pct_available
    FROM pages
    WHERE month = '2026-03'
""").df()

In [ ]:
# QUERY 4: Missing value counts for our five features (available rows only)
con.execute("""
    SELECT
        COUNT(*)                                                AS available_rows,
        SUM(CASE WHEN avg_position IS NULL THEN 1 ELSE 0 END)  AS null_avg_position,
        SUM(CASE WHEN clicks      IS NULL THEN 1 ELSE 0 END)   AS null_clicks,
        SUM(CASE WHEN impressions IS NULL THEN 1 ELSE 0 END)   AS null_impressions,
        SUM(CASE WHEN ctr         IS NULL THEN 1 ELSE 0 END)   AS null_ctr,
        SUM(CASE WHEN word_count  IS NULL THEN 1 ELSE 0 END)   AS null_word_count
    FROM pages
    WHERE month = '2026-03'
      AND available IS TRUE
""").df()

## 4. Data limits

What this data **cannot** tell us — honest limits:

1. **GSC-only early rows**: Pages that existed before Google Search Console tracking began appear with `available = FALSE`. Any pattern in those early rows is unobserved — our model is blind to pages that never got GSC coverage.

2. **Unbalanced label history**: If most `needs_refresh = TRUE` labels cluster in a specific content category or traffic tier, the model will be directional for average pages but unreliable for outliers (very high-traffic pages vs. near-zero traffic pages).

3. **Window overlap effects**: Month-level aggregates smooth over within-month spikes. A page that ranked #1 for three weeks and then crashed in week four looks "medium" in the data — we cannot detect that within-month volatility.

4. **No causal link**: A drop in `avg_position` is *observed* to correlate with `needs_refresh`, but we cannot claim the content caused the ranking drop — confounders exist (seasonality, algorithm updates, competitor changes).

5. **Anonymization ceiling**: Because URLs, titles, and keywords are stripped, we cannot build topic-aware features. A tech page and a recipe page look identical to the model if their GSC numbers match.

> All results from this notebook are **observed / measured / directional / decision-support** — not predictions of Google's algorithm.

## 5. Feature frame (5 features, mid-panel month)

Each feature is knowable at decision time — verified below.

In [ ]:
# Build a small feature frame for month=2026-03 (available rows only)
feature_frame = con.execute("""
    SELECT
        page_id,
        avg_position,   -- knowable at decision moment: measured position over the past month
        clicks,         -- knowable at decision moment: GSC clicks accumulated in the past month
        impressions,    -- knowable at decision moment: GSC impressions accumulated in the past month
        ctr,            -- knowable at decision moment: derived from clicks/impressions, both past
        word_count,     -- knowable at decision moment: measured from current published content
        needs_refresh   -- LABEL — kept here only to show distribution, NOT a feature
    FROM pages
    WHERE month = '2026-03'
      AND available IS TRUE
    LIMIT 20
""").df()

print(f"Feature frame shape (sample): {feature_frame.shape}")
feature_frame.head(10)

In [ ]:
# Feature summary statistics
full_frame = con.execute("""
    SELECT
        avg_position, clicks, impressions, ctr, word_count, needs_refresh
    FROM pages
    WHERE month = '2026-03'
      AND available IS TRUE
""").df()

full_frame.describe()

## 6. The Leakage Trap

We deliberately add a label-derived column, watch the score jump toward perfect, then delete it.

> **Lesson from notebook 02**: A feature that is derived from — or perfectly correlated with — the label gives the model "the answer" during training. The score looks great but the model is useless on new data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np

df = con.execute("""
    SELECT avg_position, clicks, impressions, ctr, word_count,
           needs_refresh
    FROM pages
    WHERE month = '2026-03'
      AND available IS TRUE
      AND avg_position IS NOT NULL
      AND clicks IS NOT NULL
      AND impressions IS NOT NULL
      AND ctr IS NOT NULL
      AND word_count IS NOT NULL
""").df()

y = df['needs_refresh'].astype(int)

# --- STEP 1: Honest model (no leakage) ---
X_honest = df[['avg_position', 'clicks', 'impressions', 'ctr', 'word_count']]
honest_score = cross_val_score(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    X_honest, y, cv=5, scoring='accuracy'
).mean()
print(f"✅ Honest model accuracy (5-fold CV): {honest_score:.3f}")

# --- STEP 2: Add a leaky column derived from the label ---
df['leaky_feature'] = df['needs_refresh'].astype(int) * 99  # directly encodes the label!
X_leaky = df[['avg_position', 'clicks', 'impressions', 'ctr', 'word_count', 'leaky_feature']]
leaky_score = cross_val_score(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    X_leaky, y, cv=5, scoring='accuracy'
).mean()
print(f"🚨 Leaky model accuracy (5-fold CV): {leaky_score:.3f}  ← suspiciously high!")

# --- STEP 3: Delete the leaky column ---
df.drop(columns=['leaky_feature'], inplace=True)
print(f"\n🗑️  Leaky column deleted. Keeping honest score: {honest_score:.3f}")
print("\nLesson: A near-perfect score on training data is a red flag, not a win.")

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.